# PRC1.3: Transfer Learning via Fine-tuning for Image Classification (TASK 1)

**Goals:**
This notebook is part of the first programming assignment (PRC1) for the course "Deep Learning for Visual Signal Processing I." In this notebook, you are required to undertake a series of tasks listed in the "2.TASKS" section.

**Learning Objectives:**
* Understand the concept of transfer learning: Learn how pre-trained deep learning models can be adapted to new classification tasks with minimal training effort.
* Implement model fine-tuning techniques: Modify and train a pre-trained model by freezing and unfreezing layers, replacing classifier heads, and optimizing hyperparameters.
* Evaluate and compare model performance: Analyze the impact of different fine-tuning strategies on model accuracy and efficiency, and explore best practices for transfer learning in deep learning applications.


**Expected Outcomes:**
* Notebooks: Generate separate notebooks for each experiment conducted during this task.
* Report: Submit a concise report (no more than two pages) that adheres to the specified course format, summarizing your findings and analyses.

**Estimated Completion Time:** The tasks are designed to be completed within an estimated timeframe of 2-3 hours when using GPU acceleration.

---

Author1: Sabbatini, Andrea (andrea.sabbatini@estudiante.uam.es)

Author2: Hamdy, Adham (adham.hamdy@estudiante.uam.es)

Author3: Ciurescu, Irina Alexandra (irinaa.ciurescu@estudiante.uam.es)

---
###### Course: Deep Learning for Visual Signal Processing I
###### Master in [Artificial Intelligence for Image Processing and Computer Vision (IPCVai)](https://ipcv.eu/)
######  [Escuela Politécnica Superior](https://www.uam.es/EPS/Home.htm), [Universidad Autónoma de Madrid](https://www.uam.es/)


# 1. Codebase

We use the provided codebase in the notebooks to develop this assignment. It contains essential scripts to access the dataset and establish the training partitions required for the tasks.

### 1-1.Install the required libraries

Check versions of installed software for this tutorial

* Python 3.10 or above
* Pytorch 2.5.1+cu121 or above
* Torchvision 0.20.1+cu121 or above

In [ ]:
import torch
import numpy as np

# Reproducibility (note: full determinism on GPU may require extra flags)
torch.manual_seed(0)
np.random.seed(0)

!python --version

import torch
print(torch.__version__)

import torchvision
print(torchvision.__version__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Python 3.12.12
2.9.0+cu126
0.24.0+cu126
Using device: cuda


### 1-2.Utility Functions (do not modify)

This cell installs and imports a set of shared utility functions used throughout the practical assignments.  
These utilities provide common functionality for:

- Model performance evaluation (overall accuracy, per-class accuracy, reports)
- Dataset inspection and class distribution analysis
- Dataset filtering and class selection
- Visualization of class imbalance

The package is maintained in a centralized GitHub repository to ensure consistency across all notebooks.  
Please **do not edit this cell**, as later sections of the tutorial rely on these functions.

In [ ]:
!pip install -q git+https://github.com/jcsma/dlvsp-utils.git

from dlvsp_utils.metrics import (
    calculate_accuracy,
    compute_accuracy_stats,
    compute_accuracy_per_class,
    print_accuracy_report,
)

from dlvsp_utils.data import (
    count_samples_dataset,
    select_classes_dataset,
    inspect_dataset_classes,
)
from dlvsp_utils.viz import (
    plot_class_distribution
)

print("✅ DLVSP utils imported: performance analysis and dataset handling ready!")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
✅ DLVSP utils imported: performance analysis and dataset handling ready!


# 2. Loading the dataset for the Classification Task
In this section we load the CIFAR-10 dataset with some basic preprocessing.

### 2-1.Basic Preprocessing
We define a basic preprocessing obtained by the tutorial.

In [ ]:
from torchvision import datasets, transforms

# Define a basic transformation when loading the dataset
transform = transforms.Compose([
    #transforms.Resize(224), #for selective fine-tuning, we will get better performance if we match the training image size of popular backbones for ImageNet (224x224)
    transforms.ToTensor(),  # Convert images to tensor format
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]#we get better performance if we match the training image normalization of popular backbones for ImageNet
    )
])

### 2-2.Load CIFAR-10
In this cell, we load the CIFAR-10 dataset (training and test splits) and apply the preprocessing.

In [ ]:
# Load CIFAR-10 data
train_full = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
test_full  = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)
class_names = train_full.classes
num_classes = len(class_names)
print(class_names)

100%|██████████| 170M/170M [00:10<00:00, 16.2MB/s] 


['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


# 3. Training from scratch ResNet50
In this section we train a ResNet50 architecture but initialized WITHOUT ImageNet weights (training the architecture from scratch). This demonstrates training the ResNet architecture from random initialization on CIFAR-10.

### 3-1.Define the ResNet Architecture
We define a `ResNet-50` model using the standard architecture **without pretrained weights**.
The final classification layer is replaced so that the network outputs predictions for the CIFAR-10 classes.

In [ ]:
import torch.nn as nn
from torchvision import models

# ResNet architecture intended to be trained from scratch (no ImageNet weights)
model_resnet = models.resnet50(weights=None) # load architecture without pre-training

# Create a new classification head for CIFAR-10
num_ftrs = model_resnet.fc.in_features
model_resnet.fc = nn.Linear(num_ftrs, num_classes)

print("✅ResNet architecture defined!")

✅ResNet architecture defined!


### 3-2.Training setup
In this cell, we define the main training components: hyperparameters, data loaders, the model, the loss function, and the optimizer.

In [ ]:
# Training setup for ResNet architecture trained from scratch (model_scratch)
import torch
import torch.optim as optim
from torch.utils.data import DataLoader

# Hyperparameters
num_workers=4
num_epochs = 10
batch_size=64
lr=0.001
weight_decay=1e-5

# Setup dataloaders
train_loader = DataLoader(train_full, batch_size=batch_size, shuffle=True, num_workers=num_workers)
test_loader = DataLoader(test_full, batch_size=batch_size, shuffle=False, num_workers=num_workers)

# Setup model, loss, and optimizer
model_resnet = model_resnet.to(device)  # instantiate custom CNN
criterion_resnet = nn.CrossEntropyLoss()
optimizer_resnet = optim.Adam(model_resnet.parameters(), lr=lr, weight_decay=weight_decay)

print("✅ Training setup ready for ResNet!")

✅ Training setup ready for ResNet!


### 3-3.Train the model
Run this cell to train **ResNet-50** (with `weights=None`, i.e., **no ImageNet pretraining**) for num_epochs epochs using the training loader. After each epoch, we switch to evaluation mode to report train and test accuracy, which helps monitor learning progress and generalization.

At the end, we print an **accuracy report per class** (train and test). This is especially useful to spot classes that the model struggles with (often visible as lower per-class accuracy, even when overall accuracy looks decent).

**Expected outcomes** (this setup, 4 CIFAR-10 classes, 10 epochs, <4 min using GPU):
* Train accuracy typically reaches around ~75–85%
* Test accuracy is usually around ~60–70%
* Some classes may remain challenging (e.g., cat vs dog), reflected in the per-class report.

In [ ]:
# Training loop for the ResNet50
print(f"Running training for ResNet network (from scratch) in {device} mode for {num_epochs} epochs")
for epoch in range(num_epochs):
    model_resnet.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer_resnet.zero_grad()
        outputs = model_resnet(images)
        loss = criterion_resnet(outputs, labels)
        loss.backward()
        optimizer_resnet.step()
        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)

    # evaluate epoch results
    model_resnet.eval()
    train_accuracy, train_labels, train_preds = calculate_accuracy(train_loader, model_resnet)
    test_accuracy, test_labels, test_preds = calculate_accuracy(test_loader, model_resnet)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}, Train Accuracy: {train_accuracy:.2f}%, Test Accuracy: {test_accuracy:.2f}%')

# Per-class accuracy
print_accuracy_report(train_labels, train_preds,
                     class_names=class_names,
                     header="TRAIN - Accuracy per class (#train samples /total):",
                     samples_per_class=count_samples_dataset(train_full, num_classes=num_classes))

print_accuracy_report(test_labels, test_preds,
                     class_names=class_names,
                     header="\nTEST - Accuracy per class (#test samples /total):",
                     samples_per_class=count_samples_dataset(test_full, num_classes=num_classes))
print("✅ Training completed!")

Running training for ResNet network (from scratch) in cuda mode for 10 epochs
Epoch [1/10], Loss: 1.8923, Train Accuracy: 44.46%, Test Accuracy: 44.46%
Epoch [2/10], Loss: 1.6099, Train Accuracy: 42.70%, Test Accuracy: 41.55%
Epoch [3/10], Loss: 1.8615, Train Accuracy: 43.42%, Test Accuracy: 43.01%
Epoch [4/10], Loss: 1.5784, Train Accuracy: 50.44%, Test Accuracy: 49.56%
Epoch [5/10], Loss: 1.3756, Train Accuracy: 56.55%, Test Accuracy: 54.54%
Epoch [6/10], Loss: 1.2737, Train Accuracy: 62.30%, Test Accuracy: 59.64%
Epoch [7/10], Loss: 1.2308, Train Accuracy: 65.81%, Test Accuracy: 62.63%
Epoch [8/10], Loss: 1.0003, Train Accuracy: 72.30%, Test Accuracy: 67.43%
Epoch [9/10], Loss: 0.9081, Train Accuracy: 74.01%, Test Accuracy: 67.75%
Epoch [10/10], Loss: 0.7791, Train Accuracy: 77.64%, Test Accuracy: 69.10%
TRAIN - Accuracy per class (#train samples /total):
Overall accuracy (micro, all samples): 77.64%
Mean per-class accuracy (macro):        77.64%

Per-class accuracy:
airplane: 80.38

# 4. Training a pretrained network
In this section, we develop a training pipeline to use a pre-trained CNN.

### 4-1.Train setup
We will run 3 different setups on the pretrained ResNet50 model:
- Feature extraction
- Selective fine-tuning
- Full fine-tuning
-
We use the same hyperparameters as before, to allow better comparison.

In [ ]:
import torch
import torch.nn as nn
from torchvision import models
import torch.optim as optim
from torch.utils.data import DataLoader

def train_setup(freeze_mode=""):
    model = models.resnet50(weights='IMAGENET1K_V2') # to load the model with IMAGENET pre-trained weights
    num_ftrs = model.fc.in_features

    if freeze_mode=="feature_extraction":
        #freeze all parameters so they become non-trainable
        for param in model.parameters():
            param.requires_grad = False
    elif freeze_mode=="fine_tune_last_block":
        for param in model.parameters():
            param.requires_grad = False
        for param in model.layer4.parameters():
            param.requires_grad = True
    else:
        for param in model.parameters():
            param.requires_grad = True

    # Create a new classification head
    # Here the size of each output sample is set to the number of classes
    model.fc = nn.Linear(num_ftrs, num_classes)
    for param in model.fc.parameters():
        param.requires_grad = True

    # Setup dataloaders
    train_loader = DataLoader(train_full, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    test_loader = DataLoader(test_full, batch_size=batch_size, shuffle=False, num_workers=num_workers)

    # Setup model, loss, and optimizer
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    params_to_update = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.Adam(params_to_update, lr=lr, weight_decay=weight_decay)

    if freeze_mode=="feature_extraction":
        print('✅ Pre-trained ResNet model loaded with frozen layers!')
    elif freeze_mode=="fine_tune_last_block":
        print('✅ Pre-trained ResNet model loaded with frozen layers except for layer4!')
    else:
        print('✅ Pre-trained ResNet model loaded!')
    print("✅ Training setup ready for ResNet!")

    return model, optimizer, criterion, train_loader, test_loader

The code below contains a function for running the training.
For each run, we will show:
- the log lines for every epoch (Loss / Train Acc / Test Acc)  
- per-class accuracies

In [ ]:
def run(model, optimizer, criterion, train_loader, test_loader):
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)

        # evaluate epoch results
        model.eval()
        train_accuracy, train_labels, train_preds = calculate_accuracy(train_loader, model)
        test_accuracy, test_labels, test_preds = calculate_accuracy(test_loader, model)
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}, Train Accuracy: {train_accuracy:.2f}%, Test Accuracy: {test_accuracy:.2f}%')

    # Per-class accuracy
    print_accuracy_report(train_labels, train_preds,
                         class_names=class_names,
                         header="TRAIN - Accuracy per class (#train samples /total):",
                         samples_per_class=count_samples_dataset(train_full, num_classes=num_classes))

    print_accuracy_report(test_labels, test_preds,
                         class_names=class_names,
                         header="\nTEST - Accuracy per class (#test samples /total):",
                         samples_per_class=count_samples_dataset(test_full, num_classes=num_classes))
    print("✅ Training completed!")

### 4-2.Feature Extraction
The pretrained backbone is frozen and only the final classification layer is trained.

In [ ]:
model, optimizer, criterion, train_loader, test_loader = train_setup("feature_extraction")
print(f"Running training for ResNet network (pretrained, feature extraction) in {device} mode for {num_epochs} epochs")
run(model, optimizer, criterion, train_loader, test_loader)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 224MB/s]


✅ Pre-trained ResNet model loaded with frozen layers!
✅ Training setup ready for ResNet!
Running training for ResNet network (pretrained, feature extraction) in cuda mode for 10 epochs
Epoch [1/10], Loss: 1.7764, Train Accuracy: 49.56%, Test Accuracy: 47.82%
Epoch [2/10], Loss: 1.6396, Train Accuracy: 50.97%, Test Accuracy: 48.09%
Epoch [3/10], Loss: 1.6224, Train Accuracy: 50.48%, Test Accuracy: 47.28%
Epoch [4/10], Loss: 1.6140, Train Accuracy: 51.17%, Test Accuracy: 47.26%
Epoch [5/10], Loss: 1.6042, Train Accuracy: 50.76%, Test Accuracy: 47.81%
Epoch [6/10], Loss: 1.6072, Train Accuracy: 51.16%, Test Accuracy: 47.69%
Epoch [7/10], Loss: 1.6116, Train Accuracy: 50.29%, Test Accuracy: 46.88%
Epoch [8/10], Loss: 1.6067, Train Accuracy: 51.32%, Test Accuracy: 47.41%
Epoch [9/10], Loss: 1.6133, Train Accuracy: 51.28%, Test Accuracy: 48.14%
Epoch [10/10], Loss: 1.6128, Train Accuracy: 50.69%, Test Accuracy: 47.35%
TRAIN - Accuracy per class (#train samples /total):
Overall accuracy (micr

### 4-3.Selective Fine-tuning
Most layers are frozen while the last block and the classifier are trained.

In [ ]:
model, optimizer, criterion, train_loader, test_loader = train_setup("fine_tune_last_block")
print(f"Running training for ResNet network (pretrained, selective fine-tuning) in {device} mode for {num_epochs} epochs")
run(model, optimizer, criterion, train_loader, test_loader)

✅ Pre-trained ResNet model loaded with frozen layers except for layer4!
✅ Training setup ready for ResNet!
Running training for ResNet network (pretrained, selective fine-tuning) in cuda mode for 10 epochs
Epoch [1/10], Loss: 1.0329, Train Accuracy: 82.66%, Test Accuracy: 74.18%
Epoch [2/10], Loss: 0.6313, Train Accuracy: 89.96%, Test Accuracy: 75.85%
Epoch [3/10], Loss: 0.4632, Train Accuracy: 94.04%, Test Accuracy: 76.60%
Epoch [4/10], Loss: 0.3539, Train Accuracy: 95.93%, Test Accuracy: 77.30%
Epoch [5/10], Loss: 0.2815, Train Accuracy: 97.08%, Test Accuracy: 76.99%
Epoch [6/10], Loss: 0.2275, Train Accuracy: 97.37%, Test Accuracy: 77.00%
Epoch [7/10], Loss: 0.2098, Train Accuracy: 98.01%, Test Accuracy: 77.15%
Epoch [8/10], Loss: 0.1866, Train Accuracy: 98.18%, Test Accuracy: 77.83%
Epoch [9/10], Loss: 0.1723, Train Accuracy: 97.67%, Test Accuracy: 76.40%
Epoch [10/10], Loss: 0.1574, Train Accuracy: 98.35%, Test Accuracy: 77.11%
TRAIN - Accuracy per class (#train samples /total):
O

### 4-4.Full Fine-tuning
All layers of the pretrained model are unfrozen and trained on the new dataset.

In [ ]:
model, optimizer, criterion, train_loader, test_loader = train_setup()
print(f"Running training for ResNet network (pretrained, full fine-tuning) in {device} mode for {num_epochs} epochs")
run(model, optimizer, criterion, train_loader, test_loader)

✅ Pre-trained ResNet model loaded!
✅ Training setup ready for ResNet!
Running training for ResNet network (pretrained, full fine-tuning) in cuda mode for 10 epochs
Epoch [1/10], Loss: 0.8786, Train Accuracy: 83.44%, Test Accuracy: 79.24%
Epoch [2/10], Loss: 0.5294, Train Accuracy: 89.48%, Test Accuracy: 83.02%
Epoch [3/10], Loss: 0.4161, Train Accuracy: 90.01%, Test Accuracy: 82.83%
Epoch [4/10], Loss: 0.3473, Train Accuracy: 91.60%, Test Accuracy: 82.28%
Epoch [5/10], Loss: 0.2888, Train Accuracy: 93.63%, Test Accuracy: 83.37%
Epoch [6/10], Loss: 0.2470, Train Accuracy: 95.53%, Test Accuracy: 85.21%
Epoch [7/10], Loss: 0.2306, Train Accuracy: 90.33%, Test Accuracy: 80.01%
Epoch [8/10], Loss: 0.2572, Train Accuracy: 95.88%, Test Accuracy: 84.25%
Epoch [9/10], Loss: 0.1568, Train Accuracy: 97.40%, Test Accuracy: 84.74%
Epoch [10/10], Loss: 0.1461, Train Accuracy: 96.81%, Test Accuracy: 84.46%
TRAIN - Accuracy per class (#train samples /total):
Overall accuracy (micro, all samples): 96.8